In [ ]:
# One-cell Phase10/11 original 646-pair SNP-level coloc replay.
# Input: original archived BBJ x GTEx V8 four-locus screen; NOT 80 new windows.
from pathlib import Path
import csv
import hashlib
import json
import shutil
import subprocess
import time
from google.colab import drive
drive.mount("/content/drive")
WORK = Path("/content/is_phase10_11_646_replay")
WORK.mkdir(parents=True,exist_ok=True)
GDRIVE = Path("/content/drive/MyDrive/MASTER_DEGREE/IS_COLAB")
INDEX = GDRIVE/"coloc_ready_v2/COLOC_ABF_INPUT_INDEX_V2.tsv"
MASTER = GDRIVE/"results_v2/COLOC_ABF_MASTER_V2.tsv"
INPUT = GDRIVE/"coloc_ready_v2/abf"
OUTPUT = GDRIVE/"results/IS_PHASE10_11_LEGACY_646_SNP_REPLAY_V1"
RUNLOG = WORK/"IS_PHASE10_11_646_R_LOG.txt"
for p in [INDEX, MASTER, INPUT]:
    if not p.exists():
        raise FileNotFoundError(f"Missing original Drive source: {p}")
if OUTPUT.exists():
    raise FileExistsError(f"Refusing existing result folder: {OUTPUT}. Use new version suffix after reviewing past results.")
def sha(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""):
            h.update(block)
    return h.hexdigest()
with INDEX.open(encoding="utf8",newline="") as handle:
    source_rows=list(csv.DictReader(handle,delimiter="\t"))
if len(source_rows)!=646:
    raise ValueError(f"Expected original 646 index rows, got {len(source_rows)}")
input_files=[INPUT/Path(x["file"]).name for x in source_rows]
n_available=sum(p.exists() and p.stat().st_size>0 for p in input_files)
print("INPUTS_INDEXED=",len(input_files))
print("INPUTS_AVAILABLE_ON_DRIVE=",n_available)
print("INPUTS_MISSING_ON_DRIVE=",646-n_available)
print("INDEX_SHA256=",sha(INDEX))
print("MASTER_SHA256=",sha(MASTER))
print("MEMORY=",Path("/proc/meminfo").read_text().splitlines()[0])
if shutil.which("Rscript") is None:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq",
                    "r-base","r-base-dev","build-essential",
                    "libcurl4-openssl-dev","libssl-dev","libxml2-dev",
                    "r-cran-data.table"],check=True)
R_INSTALL = r"""
repos <- "https://cloud.r-project.org"
if (!requireNamespace("data.table",quietly=TRUE)) {
  install.packages("data.table",repos=repos,Ncpus=2)
}
ok <- requireNamespace("coloc",quietly=TRUE) &&
      as.character(packageVersion("coloc"))=="5.2.3"
if (!ok) {
  deps <- c("ggplot2","susieR","mixsqp","Rcpp","viridis")
  need <- deps[!vapply(deps,requireNamespace,logical(1),quietly=TRUE)]
  if (length(need)>0) install.packages(need,repos=repos,Ncpus=2)
  # Prefer CRAN current exact source; fall back to its archived release.
  candidates <- c(
    "https://cran.r-project.org/src/contrib/coloc_5.2.3.tar.gz",
    "https://cran.r-project.org/src/contrib/Archive/coloc/coloc_5.2.3.tar.gz")
  installed <- FALSE
  for (url in candidates) {
    tryCatch({
      suppressWarnings(install.packages(url,repos=NULL,type="source"))
      installed <- requireNamespace("coloc",quietly=TRUE) &&
        as.character(packageVersion("coloc"))=="5.2.3"
    },error=function(e) cat("PACKAGE_SOURCE_FAILED:",conditionMessage(e),"\n"))
    if (installed) break
  }
  if (!installed) stop("Unable to pin coloc 5.2.3 from CRAN current or archive")
}
stopifnot(requireNamespace("coloc",quietly=TRUE))
stopifnot(requireNamespace("data.table",quietly=TRUE))
stopifnot(as.character(packageVersion("coloc"))=="5.2.3")
cat("COL0C_PINNED_5.2.3_READY\n")
"""
install=WORK/"install_is_646_deps.R"
install.write_text(R_INSTALL,encoding="utf8")
subprocess.run(["Rscript","--vanilla",str(install)],check=True)
R_CODE = '#!/usr/bin/env Rscript\n# Ischemic stroke Phase10/11: fail-closed, one-prior direct SNP replay of\n# ALL 646 ORIGINAL legacy GTEx gene-tissue tests (not 80 expanded windows).\n# For missing inputs, emit MISSING_INPUT; for failures, emit FAIL, never PASS.\n# This runner does not alter source files, inputs, notebooks or canonical data.\nsuppressPackageStartupMessages({\n  library(data.table)\n  library(coloc)\n})\nif (as.character(packageVersion("coloc")) != "5.2.3")\n  stop("COLOC_VERSION_PINNED_5.2.3")\nargv <- commandArgs(trailingOnly=TRUE)\nif (length(argv)!=4L)\n  stop("Usage: Rscript script.R ORIGINAL_INDEX.tsv ORIGINAL_ABF_MASTER.tsv INPUT_DIR NEW_OUTPUT_DIR")\nidx_path <- normalizePath(argv[[1]],mustWork=TRUE)\nmaster_path <- normalizePath(argv[[2]],mustWork=TRUE)\ninput_dir <- normalizePath(argv[[3]],mustWork=TRUE)\nout_dir <- argv[[4]]\nif (dir.exists(out_dir))\n  stop("REFUSE_ANY_EXISTING_OUTPUT_DIRECTORY")\nidx <- fread(idx_path)\noriginal <- fread(master_path)\nif (nrow(idx)!=646L || nrow(original)!=646L)\n  stop("LEGACY_646_DENOMINATOR_MISMATCH")\nkey <- c("locus","dataset_key","gene_base")\nneeded_idx <- c(key,"file","nsnps","qtl_n_first","qtl_n_min","qtl_n_max","status")\nneeded_orig <- c(key,"nsnps","qtl_n","status",paste0("PP.H",0:4))\nif (!all(needed_idx %in% names(idx)) || !all(needed_orig %in% names(original)))\n  stop("SOURCE_INDEX_OR_MASTER_SCHEMA_INVALID")\nif (anyDuplicated(idx,by=key) || anyDuplicated(original,by=key))\n  stop("SOURCE_COMPOSITE_KEY_NOT_UNIQUE")\njoined <- merge(idx,original,by=key,suffixes=c("_index","_old"),sort=FALSE)\nif (nrow(joined)!=646L || any(joined$status_index!="READY") ||\n    any(joined$status_old!="PASS") ||\n    any(joined$nsnps_index!=joined$nsnps_old) ||\n    any(abs(joined$qtl_n_first-joined$qtl_n)>1e-8))\n  stop("SOURCE_INDEX_MASTER_ALIGNMENT_INVALID")\nsetorder(joined,locus,dataset_key,gene_base)\nN_total <- 174686\nN_cases <- 22664\np1 <- 1e-4\np2 <- 1e-4\np12 <- 1e-5\nsource_fields <- c(key,"match_key","gwas_beta","gwas_se","gwas_maf",\n                   "eqtl_beta","eqtl_se","eqtl_maf","eqtl_n","qtl_n_scalar",\n                   "harmonization")\nrows <- vector("list",nrow(joined))\nfor (i in seq_len(nrow(joined))) {\n  s <- joined[i]\n  fname <- basename(s$file)\n  # Reject path traversal and unknown index inputs, and require name follows composite key.\n  expected <- paste0(s$locus,"__",s$dataset_key,"__",s$gene_base,".tsv")\n  if (!identical(fname,expected))\n    stop(paste("FILENAME_INDEX_MISMATCH",i,fname,expected))\n  path <- file.path(input_dir,fname)\n  stat <- "MISSING_INPUT"\n  why <- "SOURCE_FILE_NOT_PRESENT"\n  n_snps <- NA_integer_\n  seen_n_first <- NA_real_\n  n_unique <- NA_integer_\n  delta_n <- NA_real_\n  maf_big <- NA_integer_\n  harm_labels <- ""\n  pph <- rep(NA_real_,5)\n  max_diff <- NA_real_\n  has_file <- file.exists(path)\n  if (has_file) {\n    tryCatch({\n      x <- fread(path)\n      if (!all(source_fields %in% names(x)))\n        stop("SOURCE_COLUMNS_MISSING")\n      if (nrow(x)!=s$nsnps_index || uniqueN(x$match_key)!=nrow(x))\n        stop("SNP_DENOMINATOR_OR_DUPLICATE_KEY")\n      if (uniqueN(x$locus)!=1L || uniqueN(x$dataset_key)!=1L ||\n          uniqueN(x$gene_base)!=1L ||\n          x$locus[1]!=s$locus || x$dataset_key[1]!=s$dataset_key ||\n          x$gene_base[1]!=s$gene_base)\n        stop("SOURCE_COMPOSITE_KEY_MISMATCH")\n      vals <- as.matrix(x[,.(gwas_beta,gwas_se,gwas_maf,eqtl_beta,eqtl_se,\n                            eqtl_maf,eqtl_n,qtl_n_scalar)])\n      if (any(!is.finite(vals)) || any(x$gwas_se<=0) ||\n          any(x$eqtl_se<=0) || any(x$gwas_maf<=0) ||\n          any(x$eqtl_maf<=0) || any(x$gwas_maf>0.5) ||\n          any(x$eqtl_maf>0.5))\n        stop("INVALID_BETA_SE_MAF_OR_N")\n      if (uniqueN(x$qtl_n_scalar)!=1L ||\n          abs(as.numeric(x$qtl_n_scalar[1])-s$qtl_n_first)>1e-8 ||\n          abs(as.numeric(x$eqtl_n[1])-s$qtl_n_first)>1e-8)\n        stop("FIRST_QTL_N_NOT_REPRODUCIBLE")\n      if (abs(min(x$eqtl_n)-s$qtl_n_min)>1e-8 ||\n          abs(max(x$eqtl_n)-s$qtl_n_max)>1e-8)\n        stop("SNP_SPECIFIC_N_RANGE_MISMATCH")\n      # Provenance on allele orientation, NOT external cross-cohort validation.\n      harm_labels <- paste(sort(unique(as.character(x$harmonization))),collapse=";")\n      d1 <- list(beta=as.numeric(x$gwas_beta),\n                 varbeta=as.numeric(x$gwas_se)^2,\n                 snp=as.character(x$match_key),\n                 MAF=as.numeric(x$gwas_maf),\n                 N=N_total,s=N_cases/N_total,type="cc")\n      d2 <- list(beta=as.numeric(x$eqtl_beta),\n                 varbeta=as.numeric(x$eqtl_se)^2,\n                 snp=as.character(x$match_key),\n                 MAF=as.numeric(x$eqtl_maf),\n                 N=as.numeric(s$qtl_n_first),type="quant")\n      fit <- suppressMessages(suppressWarnings(\n        coloc.abf(dataset1=d1,dataset2=d2,p1=p1,p2=p2,p12=p12)\n      ))\n      new <- as.numeric(fit$summary[paste0("PP.H",0:4,".abf")])\n      old <- as.numeric(unlist(s[,paste0("PP.H",0:4),with=FALSE]))\n      if (length(new)!=5L || any(!is.finite(new)) ||\n          any(new<0) || abs(sum(new)-1)>1e-8)\n        stop("COL0C_H0_H4_INVALID")\n      max_diff <- max(abs(new-old))\n      pph <- new\n      n_snps <- nrow(x)\n      n_unique <- uniqueN(x$match_key)\n      seen_n_first <- as.numeric(x$qtl_n_scalar[1])\n      delta_n <- max(x$eqtl_n)-min(x$eqtl_n)\n      maf_big <- sum(abs(x$gwas_maf-x$eqtl_maf)>0.1)\n      if (max_diff>1e-8) stop("HISTORICAL_H0_H4_NOT_REPRODUCED")\n      stat <- "PASS"\n      why <- "H0_H4_REPLAY_MATCH_1e-8"\n    },error=function(e) {\n      stat <<- "FAIL"\n      why <<- gsub("[\\r\\n\\t]"," ",conditionMessage(e))\n    })\n  }\n  rows[[i]] <- data.table(\n    locus=s$locus,dataset_key=s$dataset_key,gene_base=s$gene_base,\n    filename=fname,status=stat,detail=why,\n    expected_nsnps=s$nsnps_index,actual_nsnps=n_snps,unique_snps=n_unique,\n    qtl_n_first_expected=s$qtl_n_first,qtl_n_first_actual=seen_n_first,\n    original_qtl_n_range=s$qtl_n_max-s$qtl_n_min,\n    observed_qtl_n_range=delta_n,maf_diff_gt_0_1=maf_big,\n    input_harmonization_labels=harm_labels,\n    PP_H0=pph[1],PP_H1=pph[2],PP_H2=pph[3],PP_H3=pph[4],\n    PP_H4=pph[5],max_abs_hypothesis_delta=max_diff,\n    model="DIRECT_SNP_ABF_ORIGINAL_FIRST_N",\n    stringsAsFactors=FALSE\n  )\n  if (i%%50L==0L || i==nrow(joined)) {\n    cat("BATCH_PROGRESS",i,"/",nrow(joined),\n        "PASS",sum(vapply(rows[seq_len(i)],function(z) z$status=="PASS",logical(1))),\n        "FAIL",sum(vapply(rows[seq_len(i)],function(z) z$status=="FAIL",logical(1))),\n        "\\n")\n    flush.console()\n  }\n}\nall <- rbindlist(rows)\nif (nrow(all)!=646L || anyDuplicated(all,by=key))\n  stop("FINAL_DENOMINATOR_MISMATCH")\ncounts <- all[, .N, by=status]\npass <- sum(all$status=="PASS")\nfailed <- sum(all$status=="FAIL")\nmissing <- sum(all$status=="MISSING_INPUT")\nif (pass+failed+missing!=646L) stop("STATUS_PARTITION_INVALID")\n# Use one new isolated output folder with all statuses; do not mislabel PARTIAL.\ndir.create(out_dir,recursive=TRUE,showWarnings=FALSE)\nfwrite(all,file.path(out_dir,"IS_LEGACY_646_SNP_REPLAY_STATUS.tsv"),\n       sep="\\t",na="NA",quote=FALSE)\nfwrite(all[status=="PASS"],file.path(out_dir,"IS_LEGACY_SNP_REPLAY_PASS_ONLY.tsv"),\n       sep="\\t",na="NA",quote=FALSE)\nwriteLines(c(\n  "# Original BBJ–GTEx legacy coloc 646-input direct-SNP replay",\n  paste0("- Status: ",if (pass==646L) "FULL_646_REPLAY_PASS" else "PARTIAL_OR_FAILED_REPLAY"),\n  paste0("- PASS: ",pass,"; FAIL: ",failed,"; MISSING_INPUT: ",missing,\n         "; total: 646"),\n  "- PASS is assigned only after an original same-SNP-input H0–H4 coloc.abf 5.2.3 rerun matches archival hypotheses within 1e-8.",\n  "- Missing input is never PASS. Errors are reported per gene/tissue; original source and output preserved.",\n  "- Original p1=p2=1e-4, p12=1e-5; BBJ case-control N=174686, cases=22664; GTEx QTL N from original FIRST scalar.",\n  "- All 646 results are from original FOUR BBJ loci; no expanded 80-window analyses have been performed here.",\n  "- No false causal/eQTL-to-stroke gene claim: cohort ancestry differences, multi-signal LD, selected-tissue multiplicity remain.",\n  "- QTL sample N varies per SNP; original FIRST N is reproduced as a historical computation, not a validated effective-N model."\n),file.path(out_dir,"IS_LEGACY_646_REPLAY_README.md"))\ncat("IS_LEGACY_646_BATCH_REPLAY=COMPLETE_INPUT_ACCOUNTING\\n")\ncat("BATCH_STATUS",if (pass==646L) "FULL_PASS" else "PARTIAL",\n    "PASS",pass,"FAIL",failed,"MISSING",missing,"\\n")\n'
script=WORK/"IS_PHASE10_11_646_batch.R"
script.write_text(R_CODE,encoding="utf8")
with script.open(encoding="utf8") as f:
    print("R_SCRIPT_SHA256=",sha(script))
subprocess.run(["Rscript","--vanilla","-e",
  "invisible(parse(file=commandArgs(trailingOnly=TRUE)[1]));cat('ORIGINAL_646_R_PARSE_OK\\n')",
  str(script)],check=True)
start=time.time()
with RUNLOG.open("w",encoding="utf8") as log:
    proc=subprocess.run(["Rscript","--vanilla",str(script),
          str(INDEX),str(MASTER),str(INPUT),str(OUTPUT)],
          stdout=log,stderr=subprocess.STDOUT,check=False)
print("COLOC_PROCESS_EXIT=",proc.returncode)
print("RUNTIME_SECONDS=",round(time.time()-start,2))
lines=RUNLOG.read_text(encoding="utf8",errors="replace").splitlines()
print("\n".join(lines[-35:]))
if proc.returncode!=0:
    raise RuntimeError(f"R replay failed (exit={proc.returncode}); inspect {RUNLOG}. Original files unchanged.")
status=OUTPUT/"IS_LEGACY_646_SNP_REPLAY_STATUS.tsv"
if not status.exists():
    raise FileNotFoundError("Runner emitted no audit status")
with status.open(encoding="utf8",newline="") as f:
    results=list(csv.DictReader(f,delimiter="\t"))
if len(results)!=646:
    raise ValueError("Final status must include exactly 646 original rows")
counts={k:sum(row["status"]==k for row in results) for k in ("PASS","FAIL","MISSING_INPUT")}
if sum(counts.values())!=646:
    raise ValueError("Status counts do not sum to 646")
# Hash each accessible SNP input after the audit. The output remains a fresh folder.
with (OUTPUT/"IS_646_SOURCE_FILE_SHA256.tsv").open("w",encoding="utf8",newline="") as audit_file:
    writer=csv.writer(audit_file,delimiter="\t")
    writer.writerow(["locus","dataset_key","gene_base","filename","status","source_sha256"])
    for entry in results:
        source_input=INPUT/entry["filename"]
        file_hash=sha(source_input) if source_input.is_file() else "MISSING"
        writer.writerow([entry["locus"],entry["dataset_key"],entry["gene_base"],
                        entry["filename"],entry["status"],file_hash])
manifest={
    "schema":"IS_PHASE10_11_646_ORIGINAL_SNP_BATCH_COLAB_V1",
    "source":"ARCHIVED_FOUR_BBJ_LOCI_GTEX_NOT_EXPANDED_80",
    "original_code_baseline_priors":{"p1":1e-4,"p2":1e-4,"p12":1e-5},
    "R_package_coloc":"5.2.3",
    "inputs_available_before_run":n_available,
    "counts":counts,
    "status":"FULL_646_NUMERIC_REPLAY" if counts["PASS"]==646 else "PARTIAL_OR_FAILED",
    "evidence_boundary":"NUMERIC_REPRODUCIBILITY_NOT_CAUSAL_GENE_OR_ANCESTRY_LD",
    "source_sha256":{"original_index":sha(INDEX),"original_master":sha(MASTER),
                     "embedded_R_script":sha(script),"output_status":sha(status)},
    "runtime_seconds":round(time.time()-start,2)}
(OUTPUT/"IS_646_COLAB_RUN_MANIFEST.json").write_text(
  json.dumps(manifest,indent=2,ensure_ascii=False)+"\n",encoding="utf8")
shutil.copyfile(RUNLOG,OUTPUT/"IS_646_FULL_RUN_LOG.txt")
print(json.dumps(manifest,indent=2,ensure_ascii=False))
print("IS_PHASE10_11_646_BATCH=RUN_ENDED_AND_STATUS_ACCOUNTED")
